In [1]:
# ===== ПОДРАЗДЕЛ 2.6 — инструменты =====
# В задании названы MathCad, MatLab и Derive. Их здесь нет, поэтому всё
# считается на Python: отделение корней — табулированием и графиком,
# уточнение — теми же методами половинного деления и простой итерации.

import math


def plot(f, x0, x1, w=68, h=19, title=""):
    xs = [x0 + (x1 - x0) * i / (w - 1) for i in range(w)]
    ys = []
    for x in xs:
        try:
            ys.append(f(x))
        except (ValueError, ZeroDivisionError):
            ys.append(None)
    good = [v for v in ys if v is not None]
    lo, hi = min(good), max(good)
    if hi == lo:
        hi = lo + 1
    pad = (hi - lo) * 0.05
    lo, hi = lo - pad, hi + pad
    row = lambda v: int(round((hi - v) / (hi - lo) * (h - 1)))
    grid = [[" "] * w for _ in range(h)]
    if lo <= 0 <= hi:
        for c in range(w):
            grid[row(0.0)][c] = "-"
    for c, v in enumerate(ys):
        if v is None:
            continue
        r = max(0, min(h - 1, row(v)))
        grid[r][c] = "*"
    if title:
        print(title)
    for r in range(h):
        print(f"{hi - (hi - lo) * r / (h - 1):+9.3f} |" + "".join(grid[r]))
    print(" " * 10 + "+" + "-" * w)
    print(" " * 11 + f"{x0:<{w // 2}.4g}{x1:>{w // 2}.4g}")


def tabulate(f, x0, x1, step, quiet=False):
    """Табулирование; возвращает отрезки со сменой знака."""
    segs, prev, x = [], None, x0
    if not quiet:
        print(f"{'x':>9}{'f(x)':>15}{'знак':>7}")
    while x <= x1 + 1e-12:
        try:
            v = f(x)
            if not quiet:
                print(f"{x:>9.3f}{v:>15.6f}{'+' if v > 0 else '−':>7}")
            if prev is not None and prev[1] * v < 0:
                segs.append((prev[0], x))
            prev = (x, v)
        except (ValueError, ZeroDivisionError):
            prev = None
        x += step
    return segs


def bisect(f, a, b, eps, show=True, name=""):
    """Половинное деление."""
    fa, n = f(a), 0
    if show:
        print(name)
        print(f"{'n':>3}{'a':>13}{'b':>13}{'c':>13}{'f(c)':>14}{'b−a':>11}")
    while b - a > eps:
        c = (a + b) / 2
        fc = f(c)
        if show:
            print(f"{n:>3}{a:>13.7f}{b:>13.7f}{c:>13.7f}{fc:>14.3e}{b - a:>11.2e}")
        if fa * fc <= 0:
            b = c
        else:
            a, fa = c, fc
        n += 1
    return (a + b) / 2, n


def iterate(phi, x0, eps, q=None, show=True, name="", cap=200):
    """Простая итерация. Если задано q, критерий строгий: q/(1−q)·|Δ| < eps."""
    if show:
        print(name)
        head = f"{'n':>3}{'xₙ':>16}{'φ(xₙ)':>16}{'|Δ|':>12}"
        print(head + (f"{'q/(1−q)·|Δ|':>15}" if q else ""))
    x = x0
    for n in range(cap):
        y = phi(x)
        d = abs(y - x)
        strict = q / (1 - q) * d if q else d
        if show:
            line = f"{n:>3}{x:>16.9f}{y:>16.9f}{d:>12.2e}"
            print(line + (f"{strict:>15.2e}" if q else ""))
        if strict < eps:
            return y, n
        x = y
    raise RuntimeError("не сошлось")


print("Инструменты готовы: plot, tabulate, bisect, iterate.")


Инструменты готовы: plot, tabulate, bisect, iterate.


In [2]:
# ===== УПРАЖНЕНИЕ 2.6, п. 1 =====
# Методом половинного деления найти корень уравнения lg x − cos x = 0
# с четырьмя знаками после запятой. Корни отделить графически.

print("=" * 78)
print("2.6.1   lg x − cos x = 0        (в задании — MathCad)")
print("=" * 78)

f1 = lambda x: math.log10(x) - math.cos(x)

print("""
ОТДЕЛЕНИЕ ГРАФИЧЕСКИ
Разносим на y₁ = lg x и y₂ = cos x. Логарифм монотонно растёт,
косинус колеблется в полосе [−1; 1] — значит пересечения возможны
только пока lg x ≤ 1, то есть при x ≤ 10. Дальше корней нет.
""")
plot(f1, 0.3, 12.0, title="f(x) = lg x − cos x   на [0,3; 12]")

print("\nТабулирование с шагом 0,5:")
segs = tabulate(f1, 0.5, 12.0, 0.5, quiet=True)
for a, b in segs:
    print(f"   смена знака на [{a:.1f}; {b:.1f}]")
print(f"\nвсего отрезков: {len(segs)} — столько же и корней")

print("\n" + "-" * 78)
print("УТОЧНЕНИЕ ПОЛОВИННЫМ ДЕЛЕНИЕМ до 4 знаков после запятой (ε = 1e-4)")
print("-" * 78)

roots = []
for i, (a, b) in enumerate(segs, 1):
    print()
    r, n = bisect(f1, a, b, 1e-4, name=f"корень {i}, отрезок [{a:.1f}; {b:.1f}]:")
    roots.append(r)
    print(f"   x{i} = {r:.4f}   (шагов {n}, невязка {f1(r):+.2e})")

print("\n" + "-" * 78)
print("ОТВЕТ:", "   ".join(f"x{i} = {r:.4f}" for i, r in enumerate(roots, 1)))
print("-" * 78)
print("""
Замечание. Первый корень лежит рядом с точкой, где косинус меняет знак,
остальные два — по краям одного «горба» косинуса. Из-за этого второй и
третий корни расположены близко (5,55 и 6,86): если взять шаг табулирования
1,0 вместо 0,5, они попадут в один отрезок и один из них потеряется.
""")


2.6.1   lg x − cos x = 0        (в задании — MathCad)

ОТДЕЛЕНИЕ ГРАФИЧЕСКИ
Разносим на y₁ = lg x и y₂ = cos x. Логарифм монотонно растёт,
косинус колеблется в полосе [−1; 1] — значит пересечения возможны
только пока lg x ≤ 1, то есть при x ≤ 10. Дальше корней нет.

f(x) = lg x − cos x   на [0,3; 12]
   +2.144 |                                                                    
   +1.933 |                                                  ******            
   +1.723 |                                                **      **          
   +1.512 |               *****                           *          **        
   +1.301 |             **     **                       **             *       
   +1.090 |            *         **                    *                *      
   +0.879 |           *            *                  *                  *     
   +0.668 |         **              *                *                    **   
   +0.458 |        *                 **             *     

In [3]:
# ===== УПРАЖНЕНИЕ 2.6, п. 2 =====
# Найти графически корни уравнения x² − 3,2x = 1 и уточнить действительный
# корень одним из итерационных методов с точностью 1e-4.

print("=" * 78)
print("2.6.2   x² − 3,2x = 1        (в задании — MathCad или MatLab)")
print("=" * 78)

f2 = lambda x: x * x - 3.2 * x - 1

print("""
Уравнение приводится к виду x² − 3,2x − 1 = 0 — оно КВАДРАТНОЕ, поэтому
корни известны точно и годятся для проверки численного ответа:
""")
D = 3.2 ** 2 + 4
xp = (3.2 + math.sqrt(D)) / 2
xm = (3.2 - math.sqrt(D)) / 2
print(f"   D = 3,2² + 4 = {D:.2f},   √D = {math.sqrt(D):.10f}")
print(f"   x₁ = (3,2 + √D)/2 = {xp:.10f}")
print(f"   x₂ = (3,2 − √D)/2 = {xm:.10f}")

print("\nОТДЕЛЕНИЕ ГРАФИЧЕСКИ:")
plot(f2, -1.5, 4.5, title="f(x) = x² − 3,2x − 1")
segs = tabulate(f2, -1.5, 4.5, 0.5, quiet=True)
for a, b in segs:
    print(f"   смена знака на [{a:.1f}; {b:.1f}]")

print("\n" + "-" * 78)
print("УТОЧНЕНИЕ ПРОСТОЙ ИТЕРАЦИЕЙ (положительный корень)")
print("-" * 78)
print("""
Из x² = 3,2x + 1 получаем φ(x) = √(3,2x + 1).
φ′(x) = 1,6/√(3,2x+1);  при x ≈ 3,49 это ≈ 0,459 < 1 — процесс сходится.
Производная ПОЛОЖИТЕЛЬНА, значит приближения подходят с одной стороны
(«лестница»), и разность соседних приближений занижает ошибку.
Поэтому применяем строгий критерий  q/(1−q)·|Δ| < ε.
""")
phi = lambda x: math.sqrt(3.2 * x + 1)
q = 1.6 / math.sqrt(3.2 * xp + 1)
print(f"   q = φ′(x*) = {q:.6f},   q/(1−q) = {q / (1 - q):.4f}\n")

r, n = iterate(phi, 3.0, 1e-4, q=q, name="Расчётная таблица, x₀ = 3:")
print(f"\n   найдено:  x = {r:.7f}   за {n + 1} итераций")
print(f"   точное:   x = {xp:.10f}")
print(f"   ошибка:   {abs(r - xp):.2e}   (требовалось < 1e-4)")
print(f"\n   ОТВЕТ: x = {r:.4f}   (второй корень x = {xm:.4f})")


2.6.2   x² − 3,2x = 1        (в задании — MathCad или MatLab)

Уравнение приводится к виду x² − 3,2x − 1 = 0 — оно КВАДРАТНОЕ, поэтому
корни известны точно и годятся для проверки численного ответа:

   D = 3,2² + 4 = 14.24,   √D = 3.7735924528
   x₁ = (3,2 + √D)/2 = 3.4867962264
   x₂ = (3,2 − √D)/2 = -0.2867962264

ОТДЕЛЕНИЕ ГРАФИЧЕСКИ:
f(x) = x² − 3,2x − 1
   +6.530 |                                                                    
   +5.943 |*                                                                   
   +5.356 | *                                                                  
   +4.769 |  *                                                                *
   +4.182 |   **                                                             * 
   +3.594 |     *                                                          **  
   +3.007 |      *                                                        *    
   +2.420 |       *                                                      *     

In [4]:
# ===== УПРАЖНЕНИЕ 2.6, п. 3 =====
# Найти наибольший положительный корень уравнения 4x − 3·ln x = 4
# с точностью 1e-4 методом итераций. Корни отделить графически.

print("=" * 78)
print("2.6.3   4x − 3·ln x = 4        (в задании — Derive)")
print("=" * 78)

f3 = lambda x: 4 * x - 3 * math.log(x) - 4

print("""
ИССЛЕДОВАНИЕ. f′(x) = 4 − 3/x обращается в нуль при x = 0,75, и это
минимум (f″ = 3/x² > 0). Значит функция сначала убывает, потом растёт,
и корней может быть не больше двух.
""")
print(f"   f(0,75) = {f3(0.75):+.6f}  < 0  →  минимум ниже оси, корня действительно два")

plot(f3, 0.2, 2.0, title="\nf(x) = 4x − 3·ln x − 4   на [0,2; 2]")

print("\nТабулирование с шагом 0,1:")
segs = tabulate(f3, 0.3, 1.6, 0.1, quiet=True)
for a, b in segs:
    print(f"   смена знака на [{a:.1f}; {b:.1f}]")

print(f"\n   f(1) = {f3(1.0):+.0f} — единица оказывается ТОЧНЫМ корнем:")
print("   4·1 − 3·ln 1 = 4 − 0 = 4.  Это и есть наибольший положительный корень,")
print("   так что численный ответ можно сверить с ним до последнего знака.")

print("\n" + "-" * 78)
print("УТОЧНЕНИЕ МЕТОДОМ ИТЕРАЦИЙ")
print("-" * 78)
print("""
Из 4x = 3·ln x + 4 получаем φ(x) = 0,75·ln x + 1.
φ′(x) = 0,75/x;  вблизи x = 1 это 0,75 < 1 — сходимость есть, но медленная.
Производная положительна: приближения идут к корню С ОДНОЙ СТОРОНЫ,
поэтому |Δ| ошибку занижает и нужен строгий критерий q/(1−q)·|Δ| < ε.
При q = 0,75 множитель q/(1−q) = 3 — то есть настоящая ошибка втрое
больше наблюдаемой разности.
""")
phi3 = lambda x: 0.75 * math.log(x) + 1
r, n = iterate(phi3, 1.5, 1e-4, q=0.75, show=False)
r_all = []
x = 1.5
for i in range(n + 1):
    y = phi3(x)
    r_all.append((i, x, y, abs(y - x)))
    x = y
print(f"{'n':>3}{'xₙ':>16}{'φ(xₙ)':>16}{'|Δ|':>12}{'3·|Δ|':>12}")
for i, a, b, d in r_all[:6]:
    print(f"{i:>3}{a:>16.9f}{b:>16.9f}{d:>12.2e}{3 * d:>12.2e}")
print(f"{'...':>3}")
for i, a, b, d in r_all[-4:]:
    print(f"{i:>3}{a:>16.9f}{b:>16.9f}{d:>12.2e}{3 * d:>12.2e}")

print(f"\n   найдено:  x = {r:.7f}   за {n + 1} итераций")
print(f"   точное:   x = 1")
print(f"   ошибка:   {abs(r - 1.0):.2e}   (требовалось < 1e-4)")

print("\n" + "-" * 78)
print("СКОЛЬКО СТОИТ МЕДЛЕННАЯ СХОДИМОСТЬ")
print("-" * 78)
r2, n2 = bisect(f3, 0.9, 1.4, 1e-4, show=False)
print(f"   простая итерация (q = 0,75):   {n + 1} шагов")
print(f"   половинное деление на [0,9; 1,4]:  {n2} шагов,  x = {r2:.7f}")
print("""
Здесь простая итерация проигрывает даже половинному делению: q = 0,75
близко к единице, и каждый шаг сокращает ошибку всего на четверть.
В упражнении 2.3 у нас было q ≈ 0,55 — и хватало 15 шагов вместо 28.
Скорость целиком определяется тем, насколько удачно выбрано φ(x).

   ОТВЕТ: наибольший положительный корень x = 1,0000
          (второй корень уравнения x ≈ 0,5456)
""")


2.6.3   4x − 3·ln x = 4        (в задании — Derive)

ИССЛЕДОВАНИЕ. f′(x) = 4 − 3/x обращается в нуль при x = 0,75, и это
минимум (f″ = 3/x² > 0). Значит функция сначала убывает, потом растёт,
и корней может быть не больше двух.

   f(0,75) = -0.136954  < 0  →  минимум ниже оси, корня действительно два

f(x) = 4x − 3·ln x − 4   на [0,2; 2]
   +2.023 |                                                                    
   +1.898 |                                                                  **
   +1.772 |                                                                **  
   +1.646 |*                                                             **    
   +1.521 |                                                            **      
   +1.395 | *                                                        **        
   +1.269 |                                                        **          
   +1.143 |  *                                                   **            
   +1.018 |        